# Context is a budget, not a bag

**The one question this notebook answers:** given *N* relevant memories and a hard budget of
*B* tokens, **what do you keep — and at what level of detail?**

Where this sits in MemoryHub:

```
task ──▶ search (ranks memories by relevance) ──▶ PACK (this notebook) ──▶ context pack ──▶ agent
                                                    └── fits in B tokens ──┘
```

Retrieval gives you an ordered pile of relevant memories. But an agent's context window is
finite, and the *most relevant* memory might be a 900-token essay while ten others are one-liners.
You cannot paste the whole pile. Packing is the decision of **what makes the cut, and how much of
each you show.**

The trick MemoryHub uses: every memory can be shown at three shrinking **levels** —

| level | what it is | cost |
|-------|------------|------|
| `full` | the whole markdown body | 💰💰💰 |
| `description` | the curated one-line summary (already in frontmatter) | 💰 |
| `title` | just the heading | ~free |

So packing isn't "in or out" — it's "in, at the richest level that still fits."

_Runs against the installed `memoryhub` package; needs `matplotlib` for the plots
(`pip install matplotlib`)._

## 1. A tiny, concrete pile of memories

No personal data — six synthetic memories with deliberately varied body sizes. We use the *real*
engine pieces (`MemoryDoc`, `render`, `count_tokens`) so what we learn here is exactly what ships.
They arrive already ranked by relevance (rank 1 = most relevant).

In [ ]:
from datetime import date

from memoryhub.models import Frontmatter, MemoryDoc
from memoryhub.bundle import render
from memoryhub.tokens import count_tokens, counter_name

print("token counter in this environment:", counter_name())


def memory(id, title, description, body):
    fm = Frontmatter(
        id=id, title=title, type="skill", description=description, tags=[],
        status="active", visibility="private",
        created=date(2026, 1, 1), updated=date(2026, 1, 2),
    )
    return MemoryDoc(frontmatter=fm, body=body)


# rank 1..6 (most relevant first); bodies range from an essay to a stub
MEMORIES = [
    memory("m-transformers", "Transformers",
           "attention, self-attention, and the encoder/decoder split",
           "The transformer replaces recurrence with self-attention. " * 30),
    memory("m-embeddings", "Embeddings",
           "dense vector representations and nearest-neighbour retrieval",
           "An embedding maps text to a vector so similar text lands nearby. " * 12),
    memory("m-quantization", "Quantization",
           "shrinking model weights to INT8/FP8 for cheaper inference",
           "Quantization stores weights in fewer bits. " * 6),
    memory("m-kv-cache", "KV cache",
           "caching attention keys/values to avoid recompute at decode time", "Cache keys and values."),
    memory("m-batching", "Continuous batching",
           "packing many requests into one GPU pass for throughput", "Batch requests together."),
    memory("m-lora", "LoRA",
           "low-rank adapters for cheap fine-tuning", "Train small adapters, freeze the base."),
]
for rank, m in enumerate(MEMORIES, 1):
    print(f"#{rank}  {m.id:<16} body={len(m.body):>4} chars")

## 2. Why levels exist: bodies are big, descriptions are tiny

Plot the token cost of each memory rendered at `full` vs `description`. This is the whole reason a
budget forces hard choices — and the whole reason the `description` tier is such a bargain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ids = [m.id for m in MEMORIES]
full_tokens = [count_tokens(render(m, "full")) for m in MEMORIES]
desc_tokens = [count_tokens(render(m, "description")) for m in MEMORIES]

x = np.arange(len(ids))
w = 0.4
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, full_tokens, w, label="full body", color="#4C78A8")
ax.bar(x + w / 2, desc_tokens, w, label="description", color="#F58518")
ax.set_xticks(x, ids, rotation=30, ha="right")
ax.set_xlabel("memory (in relevance order →)")
ax.set_ylabel("tokens")
ax.set_title("Cost per memory: full body vs one-line description")
ax.legend()
fig.tight_layout()
plt.show()

print("full-body total:", sum(full_tokens), "tokens — won't fit a small budget")
print("description total:", sum(desc_tokens), "tokens — all six, cheaply")

## 3. Three levels of one memory

Look at a single memory at each level. Notice the cost collapse `full → description → title` — the
packer will lean on exactly this to squeeze more memories in when the budget gets tight.

In [ ]:
top = MEMORIES[0]
for level in ("full", "description", "title"):
    block = render(top, level)
    preview = block if len(block) <= 70 else block[:67] + "..."
    print(f"{level:<12} {count_tokens(block):>4} tok   {preview!r}")

## ✏️ Checkpoint 1 — predict before you run

You have a **500-token budget** and the six memories above (already in relevance order).

Before running the next cells, write down your guess:
- How many memories make it in?
- Which ones get the `full` body, and which get demoted to `description` or `title`?

_(Jot it here: … )_

## 4. Build the packer (your turn)

Implement the greedy core. Walk the memories **in relevance order**; for each, take the **richest
level whose block still fits the remaining budget**; if even the title doesn't fit, skip it.
**Never exceed the budget.**

Fill in `greedy_pack`. The reference solution is in the next cell — try yours first.

In [ ]:
LEVELS = ("full", "description", "title")  # richest → leanest


def block_tokens(mem, level):
    """Tokens for one memory rendered at `level` (uses the real renderer + counter)."""
    return count_tokens(render(mem, level))


def greedy_pack(memories, budget):
    """Return a list of (memory, level, tokens): the richest level of each that fits, in order.

    Rules:
      * consider `memories` in the given (relevance) order;
      * for each, pick the first level in LEVELS whose block_tokens fits the *remaining* budget;
      * if none fits, skip the memory;
      * the sum of chosen tokens must never exceed `budget`.
    """
    chosen = []
    used = 0
    # TODO: implement me, then compare with the reference below.
    raise NotImplementedError
    return chosen

### Reference solution

(Peek only after attempting yours.) This is the same logic MemoryHub's `bundle.pack` uses,
minus the header/separator bookkeeping and the rank floor.

In [ ]:
def greedy_pack_reference(memories, budget):
    chosen = []
    used = 0
    for mem in memories:
        for level in LEVELS:
            cost = block_tokens(mem, level)
            if used + cost <= budget:
                chosen.append((mem, level, cost))
                used += cost
                break  # richest level that fit — stop degrading
    return chosen


# If you implemented greedy_pack, this line checks you against the reference:
try:
    mine = greedy_pack(MEMORIES, 500)
    assert [(m.id, lvl) for m, lvl, _ in mine] == [
        (m.id, lvl) for m, lvl, _ in greedy_pack_reference(MEMORIES, 500)
    ]
    print("✅ your greedy_pack matches the reference")
except NotImplementedError:
    print("greedy_pack is still a stub — using the reference below")

## 5. What does a 500-token budget actually keep?

Now compare against your Checkpoint 1 prediction.

In [ ]:
BUDGET = 500
packed = greedy_pack_reference(MEMORIES, BUDGET)
used = sum(t for _, _, t in packed)
print(f"budget {BUDGET} — used {used} tokens across {len(packed)} memories\n")
for rank, (m, level, tok) in enumerate(packed, 1):
    print(f"#{rank}  {m.id:<16} {level:<12} {tok:>4} tok")
assert used <= BUDGET  # the invariant: never exceed the budget

## 6. The budget as a waterfall

Watch the budget fill up, memory by memory. Each bar is one memory's cost stacked on the running
total; the dashed line is the budget ceiling the packer must never cross.

In [ ]:
labels = [f"{m.id}\n({level})" for m, level, _ in packed]
costs = [t for _, _, t in packed]
starts = np.concatenate([[0], np.cumsum(costs)[:-1]])

fig, ax = plt.subplots(figsize=(9, 4))
colors = {"full": "#4C78A8", "description": "#F58518", "title": "#54A24B"}
for i, (m, level, tok) in enumerate(packed):
    ax.bar(i, tok, bottom=starts[i], color=colors[level])
ax.axhline(BUDGET, ls="--", color="crimson", label=f"budget = {BUDGET}")
ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right", fontsize=8)
ax.set_xlabel("memory added (relevance order →)")
ax.set_ylabel("cumulative tokens")
ax.set_title("Budget waterfall: how the pack fills up")
ax.legend()
fig.tight_layout()
plt.show()

## ✏️ Checkpoint 2 — explain it back

In your own words: **why can "greedy by relevance, richest level that fits" beat "fit the most
memories"?** What does each strategy optimize, and when would fitting-the-most actually be worse?

_(Your answer: … )_

The next cell gives one way to see it.

## 7. Quality per token: two strategies

Model each memory's *information value* as **relevance × level-fidelity**, with relevance = 1/rank
(rank-1 matters most) and fidelity `full=1.0`, `description=0.5`, `title=0.2`. Compare two packers
across budgets:

- **relevance-first** (what ships): rank order, richest level that fits;
- **breadth-first**: everything at `title` to cram in as many memories as possible.

In [ ]:
FIDELITY = {"full": 1.0, "description": 0.5, "title": 0.2}
ORIG_RANK = {m.id: i for i, m in enumerate(MEMORIES, 1)}  # relevance = 1 / original rank


def value(packed):
    return sum(FIDELITY[level] * (1 / ORIG_RANK[m.id]) for m, level, _ in packed)


def breadth_first(memories, budget):
    chosen, used = [], 0
    for mem in memories:  # titles only → fit the most
        cost = block_tokens(mem, "title")
        if used + cost <= budget:
            chosen.append((mem, "title", cost))
            used += cost
    return chosen


budgets = list(range(40, 900, 20))
rel_first = [value(greedy_pack_reference(MEMORIES, b)) for b in budgets]
breadth = [value(breadth_first(MEMORIES, b)) for b in budgets]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(budgets, rel_first, label="relevance-first (ships)", color="#4C78A8", lw=2)
ax.plot(budgets, breadth, label="breadth-first (most memories)", color="#F58518", lw=2, ls="--")
ax.set_xlabel("token budget")
ax.set_ylabel("information value captured\n(Σ relevance × fidelity)")
ax.set_title("Relevance-first captures more value per token")
ax.legend()
fig.tight_layout()
plt.show()

## 8. The same idea, in the shipped engine

`memoryhub.bundle.pack` is the reference packer plus a task header, per-block separators, a
manifest, and a rank floor (memories past rank 10 never get `full`, so one essay can't starve the
rest). Here it is on our synthetic pile — note the manifest levels and that `total_tokens` ≤ budget.

In [ ]:
from memoryhub.bundle import pack

bundle = pack(MEMORIES, task="study LLM inference", budget=500)
print(f"{bundle.total_tokens}/{bundle.budget} tokens, counter={bundle.counter}\n")
for item in bundle.manifest:
    print(f"#{item.rank}  {item.id:<16} {item.level:<12} {item.tokens:>4} tok")
for ex in bundle.excluded:
    print(f"  excluded {ex.id} ({ex.reason})")
assert bundle.total_tokens <= bundle.budget

print("\n----- the pack the agent actually receives -----\n")
print(bundle.text[:600] + ("..." if len(bundle.text) > 600 else ""))

## Recap

- **Context is a budget, not a bag.** You can't paste every relevant memory; you choose.
- **Levels turn a binary choice into a dial.** `full → description → title` lets a memory be *in*
  cheaply instead of *out*.
- **Greedy, relevance-first, richest-level-that-fits** spends the budget where it buys the most —
  and the one hard rule is simply *never exceed the budget*.
- The `description` field (curated once, in frontmatter) is the summary tier — no LLM needed. A
  model-written level is the P14 extension of the `render` seam.

You built the core packer; `memoryhub.bundle.pack` is that core plus a manifest and a rank floor.